# Task 5: Interactive Business Dashboard in Streamlit

## Problem Statement & Objective
Build an **interactive Streamlit dashboard** to analyze sales, profit, and segment-wise performance using the **Global Superstore Dataset**.

### Key Goals:
- Clean and prepare the dataset
- Build a Streamlit dashboard with filters (Region, Category, Sub-Category)
- Display KPIs: **Total Sales**, **Profit**, **Top 5 Customers by Sales**

### Skills Gained:
- Business Intelligence (BI) dashboarding
- Data storytelling
- User interactivity with Streamlit
- Visual KPI analysis


In [ ]:
# Install required libraries
!pip install streamlit pandas matplotlib seaborn plotly openpyxl -q
print("✅ Libraries installed successfully!")


In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")


## 1. Dataset Description & Loading

**Global Superstore Dataset** contains retail transaction records with:
- **Order details**: Order ID, Order Date, Ship Date, Ship Mode
- **Customer info**: Customer Name, Segment, Region, Country
- **Product details**: Category, Sub-Category, Product Name
- **Financials**: Sales, Quantity, Discount, Profit

We download the dataset directly from a public source.


In [ ]:
# Download Global Superstore Dataset
import urllib.request
import os

url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/superstore.csv"
filepath = "global_superstore.csv"

if not os.path.exists(filepath):
    try:
        urllib.request.urlretrieve(url, filepath)
        print("✅ Dataset downloaded successfully!")
    except Exception as e:
        print(f"⚠️ Download failed: {e}")
        print("Creating a realistic sample dataset instead...")
        
        # Create a sample dataset if download fails
        np.random.seed(42)
        n = 1000
        regions = ['West', 'East', 'Central', 'South']
        categories = ['Technology', 'Furniture', 'Office Supplies']
        sub_cats = {
            'Technology': ['Phones', 'Computers', 'Accessories', 'Copiers'],
            'Furniture': ['Chairs', 'Tables', 'Bookcases', 'Furnishings'],
            'Office Supplies': ['Paper', 'Binders', 'Storage', 'Appliances']
        }
        segments = ['Consumer', 'Corporate', 'Home Office']
        customers = [f'Customer_{i}' for i in range(1, 101)]
        
        cat_col = np.random.choice(categories, n)
        sub_cat_col = [np.random.choice(sub_cats[c]) for c in cat_col]
        
        data = {
            'Order ID': [f'CA-{2020+i//200}-{10000+i}' for i in range(n)],
            'Order Date': pd.date_range('2020-01-01', periods=n, freq='8H'),
            'Ship Date': pd.date_range('2020-01-05', periods=n, freq='8H'),
            'Ship Mode': np.random.choice(['Standard Class', 'Second Class', 'First Class', 'Same Day'], n),
            'Customer Name': np.random.choice(customers, n),
            'Segment': np.random.choice(segments, n),
            'Country': ['United States'] * n,
            'City': np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix'], n),
            'State': np.random.choice(['New York', 'California', 'Illinois', 'Texas', 'Arizona'], n),
            'Region': np.random.choice(regions, n),
            'Category': cat_col,
            'Sub-Category': sub_cat_col,
            'Product Name': [f'Product_{np.random.randint(1,200)}' for _ in range(n)],
            'Sales': np.round(np.random.exponential(250, n), 2),
            'Quantity': np.random.randint(1, 15, n),
            'Discount': np.round(np.random.choice([0, 0.1, 0.2, 0.3, 0.4, 0.5], n), 1),
            'Profit': np.round(np.random.normal(35, 80, n), 2),
        }
        df_sample = pd.DataFrame(data)
        df_sample.to_csv(filepath, index=False)
        print(f"✅ Sample dataset created with {n} rows!")
else:
    print("✅ Dataset file already exists!")


In [ ]:
# Load the Dataset
df = pd.read_csv(filepath, encoding='latin-1')

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 3 rows:")
df.head(3)


## 2. Data Cleaning & Preprocessing


In [ ]:
# --- Data Cleaning ---

print("=== Missing Values ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values found ✅")

# Parse dates
date_cols = [c for c in df.columns if 'Date' in c or 'date' in c]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce', infer_datetime_format=True)

print(f"\n✅ Date columns parsed: {date_cols}")

# Standardise column names (strip spaces)
df.columns = df.columns.str.strip()

# Drop duplicate rows if any
before = len(df)
df.drop_duplicates(inplace=True)
print(f"\nDuplicates removed: {before - len(df)}")

# Ensure numeric types
for col in ['Sales', 'Profit', 'Quantity', 'Discount']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(subset=['Sales', 'Profit'], inplace=True)

# Derived columns
if 'Order Date' in df.columns:
    df['Year']   = df['Order Date'].dt.year
    df['Month']  = df['Order Date'].dt.month
    df['Quarter']= df['Order Date'].dt.quarter

print("\n✅ Data cleaning complete!")
print(f"Final dataset shape: {df.shape}")
df.describe()


## 3. Exploratory Data Analysis (EDA)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Global Superstore – Exploratory Data Analysis', fontsize=16, fontweight='bold')

# 1. Sales by Region
if 'Region' in df.columns:
    region_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
    axes[0,0].bar(region_sales.index, region_sales.values, color=['#2196F3','#4CAF50','#FF9800','#9C27B0'])
    axes[0,0].set_title('Total Sales by Region')
    axes[0,0].set_ylabel('Sales ($)')
    axes[0,0].tick_params(axis='x', rotation=15)

# 2. Profit by Category
if 'Category' in df.columns:
    cat_profit = df.groupby('Category')['Profit'].sum()
    colors = ['#4CAF50' if v >= 0 else '#F44336' for v in cat_profit.values]
    axes[0,1].bar(cat_profit.index, cat_profit.values, color=colors)
    axes[0,1].set_title('Total Profit by Category')
    axes[0,1].set_ylabel('Profit ($)')

# 3. Sales Distribution
axes[0,2].hist(df['Sales'], bins=50, color='#2196F3', edgecolor='white', alpha=0.8)
axes[0,2].set_title('Sales Distribution')
axes[0,2].set_xlabel('Sales ($)')
axes[0,2].set_ylabel('Frequency')
axes[0,2].set_xlim(0, df['Sales'].quantile(0.95))

# 4. Profit vs Sales Scatter
sample = df.sample(min(500, len(df)), random_state=42)
axes[1,0].scatter(sample['Sales'], sample['Profit'], alpha=0.4, color='#9C27B0', s=20)
axes[1,0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1,0].set_title('Profit vs Sales')
axes[1,0].set_xlabel('Sales ($)')
axes[1,0].set_ylabel('Profit ($)')

# 5. Sales by Segment
if 'Segment' in df.columns:
    seg = df.groupby('Segment')['Sales'].sum()
    axes[1,1].pie(seg.values, labels=seg.index, autopct='%1.1f%%',
                  colors=['#2196F3','#FF9800','#4CAF50'], startangle=90)
    axes[1,1].set_title('Sales by Segment')

# 6. Monthly Sales Trend
if 'Year' in df.columns:
    monthly = df.groupby(['Year','Month'])['Sales'].sum().reset_index()
    monthly['Period'] = monthly['Year'].astype(str) + '-' + monthly['Month'].astype(str).str.zfill(2)
    monthly_sorted = monthly.sort_values('Period').tail(24)
    axes[1,2].plot(range(len(monthly_sorted)), monthly_sorted['Sales'], 
                   color='#2196F3', linewidth=2, marker='o', markersize=4)
    axes[1,2].set_title('Monthly Sales Trend (last 24 periods)')
    axes[1,2].set_ylabel('Sales ($)')
    axes[1,2].tick_params(axis='x', rotation=45)
    step = max(1, len(monthly_sorted)//6)
    axes[1,2].set_xticks(range(0, len(monthly_sorted), step))
    axes[1,2].set_xticklabels(monthly_sorted['Period'].iloc[::step].values, rotation=45)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ EDA visualisations saved as eda_overview.png")


## 4. KPI Analysis – Sales, Profit & Top Customers


In [ ]:
# ── KPI Summary ──
total_sales   = df['Sales'].sum()
total_profit  = df['Profit'].sum()
profit_margin = (total_profit / total_sales * 100) if total_sales else 0
total_orders  = df['Order ID'].nunique() if 'Order ID' in df.columns else len(df)

print("=" * 50)
print("          📊  KEY PERFORMANCE INDICATORS")
print("=" * 50)
print(f"  💰 Total Sales   : ${total_sales:>12,.2f}")
print(f"  📈 Total Profit  : ${total_profit:>12,.2f}")
print(f"  📉 Profit Margin : {profit_margin:>11.2f}%")
print(f"  🛒 Total Orders  : {total_orders:>12,}")
print("=" * 50)

# ── Top 5 Customers by Sales ──
if 'Customer Name' in df.columns:
    top5 = df.groupby('Customer Name')['Sales'].sum().sort_values(ascending=False).head(5)
    print("\n🏆 Top 5 Customers by Sales:")
    for i, (name, val) in enumerate(top5.items(), 1):
        print(f"  {i}. {name:<30} ${val:>10,.2f}")

    # Visualise
    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(top5.index[::-1], top5.values[::-1], color='#2196F3')
    ax.bar_label(bars, labels=[f'${v:,.0f}' for v in top5.values[::-1]], padding=5)
    ax.set_title('Top 5 Customers by Sales', fontsize=14, fontweight='bold')
    ax.set_xlabel('Total Sales ($)')
    ax.set_xlim(0, top5.max() * 1.15)
    plt.tight_layout()
    plt.savefig('top5_customers.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("\n✅ Top 5 customers chart saved as top5_customers.png")


In [ ]:
# Sub-Category Performance
if 'Sub-Category' in df.columns:
    subcat = df.groupby('Sub-Category').agg(
        Total_Sales=('Sales', 'sum'),
        Total_Profit=('Profit', 'sum'),
        Orders=('Sales', 'count')
    ).sort_values('Total_Sales', ascending=False)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Sub-Category Performance', fontsize=14, fontweight='bold')
    
    # Sales
    colors_s = ['#2196F3'] * len(subcat)
    axes[0].barh(subcat.index[::-1], subcat['Total_Sales'][::-1], color=colors_s)
    axes[0].set_title('Sales by Sub-Category')
    axes[0].set_xlabel('Total Sales ($)')
    
    # Profit
    colors_p = ['#4CAF50' if v >= 0 else '#F44336' for v in subcat['Total_Profit'][::-1]]
    axes[1].barh(subcat.index[::-1], subcat['Total_Profit'][::-1], color=colors_p)
    axes[1].axvline(0, color='black', linewidth=0.8)
    axes[1].set_title('Profit by Sub-Category')
    axes[1].set_xlabel('Total Profit ($)')
    
    plt.tight_layout()
    plt.savefig('subcat_performance.png', dpi=120, bbox_inches='tight')
    plt.show()
    
    print("\n📊 Sub-Category Summary:")
    print(subcat.to_string())


In [ ]:
# Save cleaned dataset for the Streamlit app
df.to_csv('cleaned_superstore.csv', index=False)
print("✅ Cleaned dataset saved as cleaned_superstore.csv")
print(f"   Shape: {df.shape}")


## 5. Streamlit Dashboard

The cell below writes the complete Streamlit app to `dashboard_app.py`.  
Run it with:
```bash
streamlit run dashboard_app.py
```
or use the **Launch** cell further below.


In [ ]:
streamlit_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# ── Page Config ──────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Global Superstore Dashboard",
    page_icon="🏪",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ── Load Data ─────────────────────────────────────────────────────────────────
@st.cache_data
def load_data():
    df = pd.read_csv("cleaned_superstore.csv", parse_dates=["Order Date"], 
                     infer_datetime_format=True, on_bad_lines="skip")
    return df

df = load_data()

# ── Sidebar Filters ───────────────────────────────────────────────────────────
st.sidebar.title("🔍 Filters")
st.sidebar.markdown("---")

# Region
regions = sorted(df["Region"].dropna().unique()) if "Region" in df.columns else []
sel_region = st.sidebar.multiselect("🌎 Region", options=regions, default=regions)

# Category
categories = sorted(df["Category"].dropna().unique()) if "Category" in df.columns else []
sel_cat = st.sidebar.multiselect("📦 Category", options=categories, default=categories)

# Sub-Category (dynamic based on category)
if "Sub-Category" in df.columns:
    available_sub = sorted(df[df["Category"].isin(sel_cat)]["Sub-Category"].dropna().unique())
    sel_subcat = st.sidebar.multiselect("🏷️ Sub-Category", options=available_sub, default=available_sub)
else:
    sel_subcat = []

# Segment
if "Segment" in df.columns:
    segments = sorted(df["Segment"].dropna().unique())
    sel_seg = st.sidebar.multiselect("👥 Segment", options=segments, default=segments)
else:
    sel_seg = []

# ── Apply Filters ─────────────────────────────────────────────────────────────
filt = df.copy()
if sel_region and "Region" in filt.columns:
    filt = filt[filt["Region"].isin(sel_region)]
if sel_cat and "Category" in filt.columns:
    filt = filt[filt["Category"].isin(sel_cat)]
if sel_subcat and "Sub-Category" in filt.columns:
    filt = filt[filt["Sub-Category"].isin(sel_subcat)]
if sel_seg and "Segment" in filt.columns:
    filt = filt[filt["Segment"].isin(sel_seg)]

# ── Header ────────────────────────────────────────────────────────────────────
st.title("🏪 Global Superstore Interactive Dashboard")
st.markdown(f"Showing **{len(filt):,}** records based on current filters.")
st.markdown("---")

# ── KPI Cards ─────────────────────────────────────────────────────────────────
total_sales   = filt["Sales"].sum()
total_profit  = filt["Profit"].sum()
margin        = (total_profit / total_sales * 100) if total_sales else 0
total_orders  = filt["Order ID"].nunique() if "Order ID" in filt.columns else len(filt)

k1, k2, k3, k4 = st.columns(4)
k1.metric("💰 Total Sales",   f"${total_sales:,.0f}")
k2.metric("📈 Total Profit",  f"${total_profit:,.0f}")
k3.metric("📉 Profit Margin", f"{margin:.1f}%")
k4.metric("🛒 Total Orders",  f"{total_orders:,}")

st.markdown("---")

# ── Row 1: Sales by Region & Category Profit ───────────────────────────────────
col1, col2 = st.columns(2)

with col1:
    st.subheader("🌎 Sales by Region")
    if "Region" in filt.columns:
        reg = filt.groupby("Region")["Sales"].sum().reset_index().sort_values("Sales", ascending=False)
        fig1 = px.bar(reg, x="Region", y="Sales", color="Region",
                      color_discrete_sequence=px.colors.qualitative.Set2,
                      labels={"Sales": "Total Sales ($)"},
                      text_auto=".2s")
        fig1.update_layout(showlegend=False, plot_bgcolor="rgba(0,0,0,0)")
        st.plotly_chart(fig1, use_container_width=True)

with col2:
    st.subheader("📦 Profit by Category")
    if "Category" in filt.columns:
        cat = filt.groupby("Category")["Profit"].sum().reset_index()
        cat["Color"] = cat["Profit"].apply(lambda x: "Profit" if x >= 0 else "Loss")
        fig2 = px.bar(cat, x="Category", y="Profit", color="Color",
                      color_discrete_map={"Profit": "#4CAF50", "Loss": "#F44336"},
                      text_auto=".2s",
                      labels={"Profit": "Total Profit ($)"})
        fig2.update_layout(plot_bgcolor="rgba(0,0,0,0)")
        st.plotly_chart(fig2, use_container_width=True)

# ── Row 2: Sales Trend & Sub-Category ─────────────────────────────────────────
col3, col4 = st.columns(2)

with col3:
    st.subheader("📅 Monthly Sales Trend")
    if "Order Date" in filt.columns:
        filt["YearMonth"] = filt["Order Date"].dt.to_period("M").astype(str)
        trend = filt.groupby("YearMonth")["Sales"].sum().reset_index().sort_values("YearMonth")
        fig3 = px.line(trend, x="YearMonth", y="Sales",
                       markers=True, color_discrete_sequence=["#2196F3"],
                       labels={"YearMonth": "Month", "Sales": "Total Sales ($)"})
        fig3.update_layout(plot_bgcolor="rgba(0,0,0,0)")
        fig3.update_xaxes(tickangle=45)
        st.plotly_chart(fig3, use_container_width=True)

with col4:
    st.subheader("🏷️ Sales by Sub-Category")
    if "Sub-Category" in filt.columns:
        sub = filt.groupby("Sub-Category")["Sales"].sum().reset_index().sort_values("Sales")
        fig4 = px.bar(sub, x="Sales", y="Sub-Category", orientation="h",
                      color="Sales", color_continuous_scale="Blues",
                      text_auto=".2s",
                      labels={"Sales": "Total Sales ($)"})
        fig4.update_layout(plot_bgcolor="rgba(0,0,0,0)", coloraxis_showscale=False)
        st.plotly_chart(fig4, use_container_width=True)

# ── Row 3: Top 5 Customers & Segment Pie ─────────────────────────────────────
col5, col6 = st.columns(2)

with col5:
    st.subheader("🏆 Top 5 Customers by Sales")
    if "Customer Name" in filt.columns:
        top5 = (filt.groupby("Customer Name")["Sales"]
                    .sum().sort_values(ascending=False).head(5).reset_index())
        fig5 = px.bar(top5, x="Sales", y="Customer Name", orientation="h",
                      color="Sales", color_continuous_scale="Greens",
                      text_auto=".2s",
                      labels={"Sales": "Total Sales ($)", "Customer Name": "Customer"})
        fig5.update_layout(plot_bgcolor="rgba(0,0,0,0)", coloraxis_showscale=False,
                           yaxis={"autorange": "reversed"})
        st.plotly_chart(fig5, use_container_width=True)

with col6:
    st.subheader("👥 Sales by Segment")
    if "Segment" in filt.columns:
        seg = filt.groupby("Segment")["Sales"].sum().reset_index()
        fig6 = px.pie(seg, names="Segment", values="Sales",
                      color_discrete_sequence=px.colors.qualitative.Pastel,
                      hole=0.4)
        fig6.update_traces(textposition="inside", textinfo="percent+label")
        st.plotly_chart(fig6, use_container_width=True)

# ── Profit vs Sales Scatter ───────────────────────────────────────────────────
st.subheader("🔵 Profit vs Sales (Scatter)")
col_opts = [c for c in ["Category", "Segment", "Region"] if c in filt.columns]
color_by = st.selectbox("Color by:", col_opts) if col_opts else None
sample = filt.sample(min(2000, len(filt)), random_state=42) if len(filt) > 2000 else filt
fig7 = px.scatter(sample, x="Sales", y="Profit", color=color_by,
                  opacity=0.6, size_max=8,
                  color_discrete_sequence=px.colors.qualitative.Set1,
                  labels={"Sales": "Sales ($)", "Profit": "Profit ($)"})
fig7.add_hline(y=0, line_dash="dot", line_color="red", annotation_text="Break-even")
fig7.update_layout(plot_bgcolor="rgba(0,0,0,0)")
st.plotly_chart(fig7, use_container_width=True)

# ── Data Table ────────────────────────────────────────────────────────────────
with st.expander("📋 View Filtered Data"):
    show_cols = [c for c in ["Order ID", "Order Date", "Customer Name", "Region",
                              "Category", "Sub-Category", "Segment", "Sales", "Profit"]
                 if c in filt.columns]
    st.dataframe(filt[show_cols].sort_values("Sales", ascending=False).reset_index(drop=True),
                 height=350, use_container_width=True)
    csv = filt[show_cols].to_csv(index=False).encode("utf-8")
    st.download_button("⬇️ Download Filtered CSV", csv, "filtered_superstore.csv", "text/csv")

st.markdown("---")
st.caption("📊 Global Superstore Dashboard | Built with Streamlit & Plotly | DevelopersHub Data Science Internship – Task 5")
'''

with open("dashboard_app.py", "w") as f:
    f.write(streamlit_code)

print("✅ Streamlit dashboard written to dashboard_app.py")
print("\\nDashboard features:")
print("  • Sidebar filters: Region, Category, Sub-Category, Segment")
print("  • KPI cards: Total Sales, Profit, Margin, Orders")
print("  • Charts: Region sales, Category profit, Monthly trend,")
print("            Sub-category sales, Top 5 Customers, Segment pie,")
print("            Profit vs Sales scatter")
print("  • Filterable data table with CSV download")


## 6. Launch the Dashboard

Run the cell below to start the Streamlit server and get a shareable link via **localtunnel**.


In [ ]:
import subprocess, threading, time, urllib.request

def run_streamlit():
    subprocess.Popen(["streamlit", "run", "dashboard_app.py",
                      "--server.port=8501", "--server.headless=true",
                      "--server.enableCORS=false"])

thread = threading.Thread(target=run_streamlit)
thread.start()
time.sleep(4)

try:
    # Install localtunnel for a public URL
    subprocess.run(["npm", "install", "-g", "localtunnel"], capture_output=True)
    lt = subprocess.Popen(["lt", "--port", "8501"], stdout=subprocess.PIPE, text=True)
    time.sleep(3)
    url_line = lt.stdout.readline().strip()
    print(f"\n🌐 Dashboard is LIVE at: {url_line}")
    print("   (If the URL doesn't appear, open http://localhost:8501 in your browser)")
except Exception as e:
    print("\n✅ Streamlit started!")
    print("   Open your browser at: http://localhost:8501")
    print(f"   (localtunnel not available: {e})")


## 7. Conclusion & Insights

### 📌 Key Findings from the Global Superstore Dashboard:

1. **Sales Performance**  
   - Regions differ significantly in revenue; the West typically leads in total sales.  
   - Technology is often the highest revenue category, while Office Supplies has the most orders.

2. **Profitability**  
   - Some sub-categories (e.g., Tables, Bookcases) may generate negative profit despite high sales — driven by heavy discounting.  
   - Copiers and Accessories tend to have healthy profit margins.

3. **Top Customers**  
   - A small group of customers drive a disproportionate share of revenue, typical of the Pareto principle.  
   - Identifying and nurturing these customers is high-priority for retention.

4. **Segment Analysis**  
   - The Consumer segment contributes the largest share of total sales.  
   - Corporate customers tend to place larger, higher-value orders.

5. **Trend Insights**  
   - Q4 consistently sees sales spikes (holiday/year-end purchasing cycles).  
   - Monthly trends can help plan inventory and marketing campaigns.

### ✅ Skills Demonstrated:
- Data cleaning and preprocessing  
- Exploratory Data Analysis with Matplotlib & Seaborn  
- Interactive dashboard development with **Streamlit** and **Plotly**  
- KPI computation and business storytelling  
- User interactivity via dynamic filters
